# Text Generation with Markov Chains

**Course:** Artificial Intelligence · Unit 6 — Natural Language Processing  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 24 — Deep Learning for Natural Language Processing  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Understand Markov chain language models as a baseline for text generation
2. Build a bigram transition dictionary from a comedy-transcript corpus
3. Sample from the model to generate novel sentences in the style of the training text
4. Identify the limitations of n-gram models and motivate neural alternatives


# Text Generation

## Introduction

Markov chains can be used for very basic text generation. Think about every word in a corpus as a state. We can make a simple assumption that the next word is only dependent on the previous word - which is the basic assumption of a Markov chain.

Markov chains don't generate text as well as deep learning, but it's a good (and fun!) start.

## Select Text to Imitate

In this notebook, we're specifically going to generate text in the style of Ali Wong, so as a first step, let's extract the text from her comedy routine.

In [ ]:
# Read in the corpus, including punctuation!
from pathlib import Path
import pandas as pd

data = pd.read_pickle(Path.cwd() / 'pickle' / 'corpus.pkl')
data

In [ ]:
# Extract only Ali Wong's text
ali_text = data.transcript.loc['dave']
ali_text[:1000]

> **Output interpretation:** The first 1000 characters of the transcript confirm the data loaded correctly. Note the style: informal language, sentence fragments, and spoken-word patterns. These idiosyncrasies will be reflected in the generated text.


## Build a Markov Chain Function

We are going to build a simple Markov chain function that creates a dictionary:
* The keys should be all of the words in the corpus
* The values should be a list of the words that follow the keys

In [ ]:
from collections import defaultdict

def markov_chain(text):
    '''The input is a string of text and the output will be a dictionary with each word as
       a key and each value as the list of words that come after the key in the text.'''
    
    # Tokenize the text by word, though including punctuation
    words = text.split(' ')
    
    # Initialize a default dictionary to hold all of the words and next words
    m_dict = defaultdict(list)
    
    # Create a zipped list of all of the word pairs and put them in word: list of next words format
    for current_word, next_word in zip(words[0:-1], words[1:]):
        m_dict[current_word].append(next_word)

    # Convert the default dict back into a dictionary
    m_dict = dict(m_dict)
    return m_dict

In [ ]:
# Create the dictionary for Ali's routine, take a look at it
ali_dict = markov_chain(ali_text)
ali_dict

> **Output interpretation:** The Markov chain dictionary maps each word to the list of words that follow it in the corpus. Words that appear in many different contexts have long value lists; rare words may map to only one or two successors. The chain captures local word order but has no memory beyond one step.


## Create a Text Generator

We're going to create a function that generates sentences. It will take two things as inputs:
* The dictionary you just created
* The number of words you want generated

Here are some examples of generated sentences:

>'Shape right turn– I also takes so that she’s got women all know that snail-trail.'

>'Optimum level of early retirement, and be sure all the following Tuesday… because it’s too.'

In [ ]:
import random

def generate_sentence(chain, count=30):
    '''Input a dictionary in the format of key = current word, value = list of next words
       along with the number of words you would like to see in your generated sentence.'''

    # Capitalize the first word
    word1 = random.choice(list(chain.keys()))
    sentence = word1.capitalize()

    # Generate the second word from the value list. Set the new word as the first word. Repeat.
    for i in range(count-1):
        word2 = random.choice(chain[word1])
        word1 = word2
        sentence += ' ' + word2

    # End it with a period
    sentence += '.'
    return(sentence)

In [ ]:
generate_sentence(ali_dict)

> **Output interpretation:** Each call to `generate_sentence()` samples a different random walk through the chain, producing a unique 30-word sequence. The output should sound vaguely like the training text but will be grammatically incoherent — a characteristic limitation of unigram/bigram Markov models. Neural language models (GPT, LLaMA) overcome this by conditioning on much longer context windows.


## Additional Exercises

1. Try making the generate_sentence function better. Maybe allow it to end with a random punctuation mark or end whenever it gets to a word that already ends with a punctuation mark.